# Agentic AI II: LangGraph & Multi-Agent Systems

Day 6 of Week 1. The hand-rolled ReAct loop from [notebook 05](/courses/llm-eng/05-agents-foundations.html) works for simple tasks but breaks down at scale: complex tasks require branching logic, multiple specialized agents operating in parallel, human approval gates, and reliable state that survives failures. LangGraph models agent behavior as a directed graph — nodes are processing steps, edges are transitions, and a typed state object flows through the graph. This makes control flow explicit, testable, and debuggable in ways that a while-loop is not.

The central design insight of LangGraph is that complex agent behavior is better expressed as a graph than as nested conditionals in a loop. When a product manager asks "what does this agent do when it encounters a high-risk document?", you can show them a diagram with a labeled edge. You cannot show them a while-loop.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()
llm_strong = LLMClient(model="gpt-4o")

## From Loops to Graphs

**Why the while-loop agent fails at scale.** The `run_agent` function from [notebook 05](/courses/llm-eng/05-agents-foundations.html) has three structural limitations:

- **No conditional branching.** A compliance review needs different paths for high-risk vs. low-risk documents: high-risk triggers GPT-4o with extended retrieval; low-risk uses GPT-4o-mini with a single retrieved passage. A while-loop with nested `if` statements implementing this quickly becomes unreadable and untestable.

- **No parallelism.** Some questions require both a filing lookup and a calculation simultaneously. The while-loop is sequential by definition — it processes one tool call at a time.

- **No persistence.** If the agent crashes at step 7 of a 10-step process, it must restart from step 1. LangGraph's checkpointing saves the typed state after every node, so recovery means resuming from the last checkpoint.

<br>

**Three LangGraph concepts.** (1) `StateGraph`: the graph object. You add nodes and edges, then call `.compile()` to get a runnable. (2) `TypedDict` state: a typed dictionary that flows through every node. Each node receives the current state and returns a partial update — only the keys it modifies. (3) `add_conditional_edges`: maps a routing function's return value to the next node name, implementing branching.

## LangGraph Fundamentals

We build a minimal 2-node graph that replicates the ReAct loop to establish the LangGraph API before building the compliance pipeline:

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
import json


class MinimalAgentState(TypedDict):
    messages: Annotated[list, add_messages]  # <1>
    step_count: int


def call_model_node(state: MinimalAgentState) -> dict:
    """Call the LLM with the current message history."""
    resp = llm._client.chat.completions.create(
        model="gpt-4o-mini",
        messages=state["messages"],
        temperature=0.0,
    )
    return {
        "messages": [resp.choices[0].message],
        "step_count": state["step_count"] + 1,
    }


def should_continue(state: MinimalAgentState) -> str:  # <2>
    """Route to END if the model is done, otherwise loop back."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "execute_tools"
    return END


def execute_tools_node(state: MinimalAgentState) -> dict:
    """Execute all requested tool calls and append results."""
    last_message = state["messages"][-1]
    tool_messages = []
    for tc in last_message.tool_calls:
        # Stub: echo the arguments back as the result
        tool_messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": f"Tool {tc.function.name} result: {tc.function.arguments}",
        })
    return {"messages": tool_messages}


# Build the graph
builder = StateGraph(MinimalAgentState)
builder.add_node("call_model", call_model_node)
builder.add_node("execute_tools", execute_tools_node)
builder.set_entry_point("call_model")
builder.add_conditional_edges("call_model", should_continue)  # <3>
builder.add_edge("execute_tools", "call_model")
minimal_graph = builder.compile()

print("Graph compiled successfully.")
print(minimal_graph.get_graph().draw_mermaid())

1. `Annotated[list, add_messages]` is LangGraph's message accumulator: when a node returns `{"messages": [...]}`, LangGraph appends to the existing list rather than replacing it — exactly the behavior we need for conversational state.
2. The routing function `should_continue` returns either `"execute_tools"` (a node name) or `END` (the special terminal constant). This drives `add_conditional_edges`.
3. `add_conditional_edges("call_model", should_continue)` means: after running `call_model`, call `should_continue(state)` and go to whichever node name it returns. This is the branching mechanism that replaces the `if finish_reason == "stop"` check in the while-loop.

## Compliance Review Pipeline

We build the compliance review graph with four nodes and a conditional edge that routes high-risk documents to GPT-4o and low-risk documents to GPT-4o-mini — a cost optimization that is trivially expressed as a graph edge but would require messy conditional logic in a while-loop.

In [ ]:
from typing import Literal


class ComplianceReport(BaseModel):
    risk_tier: Literal["high", "medium", "low"]
    flagged_items: list[str]
    citations: list[str]
    recommendation: str


class DocClassification(BaseModel):
    doc_type: Literal["contract", "disclosure", "regulatory_filing", "other"]
    risk_tier: Literal["high", "medium", "low"]
    reasoning: str


class ComplianceState(TypedDict):
    document_text: str
    doc_type: str
    risk_tier: str
    retrieved_policies: list[str]
    review_text: str
    report: Optional[ComplianceReport]


# Minimal policy store (reuse corpus from notebook 07)
POLICIES = {
    "contract": "Contracts must include governing law, dispute resolution, and liability caps per firm policy CL-001.",
    "disclosure": "Disclosures must comply with SEC Regulation S-K and include all material risk factors.",
    "regulatory_filing": "Regulatory filings must meet FINRA Rule 4110, Basel III, and CET1 requirements per firm policy RF-005.",
    "other": "All documents must comply with firm's general legal and compliance standards.",
}


def classify_node(state: ComplianceState) -> dict:
    """Classify document type and risk tier."""
    messages = [
        {"role": "system", "content": "Classify the document type and risk tier. Be concise."},
        {"role": "user", "content": f"Document:\n{state['document_text'][:1000]}"},
    ]
    result: DocClassification = llm.complete(messages, response_format=DocClassification)
    return {"doc_type": result.doc_type, "risk_tier": result.risk_tier}


def retrieve_node(state: ComplianceState) -> dict:  # <1>
    """Fetch relevant policy passages for the document type."""
    policy = POLICIES.get(state["doc_type"], POLICIES["other"])
    return {"retrieved_policies": [policy]}


def review_high_risk_node(state: ComplianceState) -> dict:
    """Full compliance review using GPT-4o for high-risk documents."""
    context = "\n".join(state["retrieved_policies"])
    messages = [
        {"role": "system", "content": f"You are a senior compliance reviewer. Relevant policies:\n{context}"},
        {"role": "user", "content": f"Review this document for compliance issues:\n{state['document_text']}"},
    ]
    report: ComplianceReport = llm_strong.complete(messages, response_format=ComplianceReport)  # <2>
    return {"report": report, "review_text": f"High-risk review by gpt-4o: {report.recommendation}"}


def review_low_risk_node(state: ComplianceState) -> dict:
    """Lightweight compliance review using GPT-4o-mini for low-risk documents."""
    context = "\n".join(state["retrieved_policies"])
    messages = [
        {"role": "system", "content": f"You are a compliance reviewer. Relevant policies:\n{context}"},
        {"role": "user", "content": f"Review this document for compliance issues:\n{state['document_text']}"},
    ]
    report: ComplianceReport = llm.complete(messages, response_format=ComplianceReport)
    return {"report": report, "review_text": f"Low-risk review by gpt-4o-mini: {report.recommendation}"}


def route_by_risk(state: ComplianceState) -> str:  # <3>
    """Route high-risk documents to GPT-4o, others to GPT-4o-mini."""
    return "review_high_risk" if state["risk_tier"] == "high" else "review_low_risk"

1. In the production pipeline from [notebook 07](/courses/llm-eng/07-rag-pipeline.html), `retrieve_node` calls `hybrid_retrieve` against the ChromaDB collection. Here we use a simple dict lookup to keep the example self-contained.
2. High-risk documents use `llm_strong` (GPT-4o) for the review. This costs roughly 15× more per call than GPT-4o-mini but provides significantly better compliance analysis — a worthwhile trade-off for documents flagged as high-risk.
3. The routing function returns a string that must match a node name registered in the graph. The `add_conditional_edges` call maps this routing function's output to the graph edges.

We assemble and compile the compliance pipeline graph:

In [ ]:
compliance_builder = StateGraph(ComplianceState)
compliance_builder.add_node("classify", classify_node)
compliance_builder.add_node("retrieve", retrieve_node)
compliance_builder.add_node("review_high_risk", review_high_risk_node)
compliance_builder.add_node("review_low_risk", review_low_risk_node)

compliance_builder.set_entry_point("classify")
compliance_builder.add_edge("classify", "retrieve")
compliance_builder.add_conditional_edges(
    "retrieve",
    route_by_risk,
    {"review_high_risk": "review_high_risk", "review_low_risk": "review_low_risk"},
)
compliance_builder.add_edge("review_high_risk", END)
compliance_builder.add_edge("review_low_risk", END)

compliance_graph = compliance_builder.compile()
print(compliance_graph.get_graph().draw_mermaid())

Running the compliance pipeline on a sample contract with a buried arbitration clause:

In [ ]:
SAMPLE_CONTRACT = """LOAN AGREEMENT dated 1 January 2025

1. Parties: Lender (First Capital Bank) and Borrower (Acme Corp).
2. Principal Amount: $5,000,000 at 7.5% per annum.
3. Governing Law: The laws of the State of New York shall govern.
4. Dispute Resolution: Any dispute shall be resolved by binding arbitration
   under the AAA Commercial Arbitration Rules. BORROWER WAIVES ALL RIGHTS TO
   CLASS ACTION PROCEEDINGS AND JURY TRIAL.
5. Liability Cap: Lender's liability shall not exceed the principal amount.
6. Indemnification: Borrower indemnifies Lender against all losses arising
   from Borrower's breach, including unlimited consequential damages."""

initial_state: ComplianceState = {
    "document_text": SAMPLE_CONTRACT,
    "doc_type": "",
    "risk_tier": "",
    "retrieved_policies": [],
    "review_text": "",
    "report": None,
}

final_state = compliance_graph.invoke(initial_state)
report: ComplianceReport = final_state["report"]

print(f"Doc type:     {final_state['doc_type']}")
print(f"Risk tier:    {final_state['risk_tier']}")
print(f"Flagged items:")
for item in report.flagged_items:
    print(f"  - {item}")
print(f"Recommendation: {report.recommendation}")

## Supervisor / Worker Pattern

For questions that span multiple knowledge domains, a single agent often performs worse than a specialized team. The **supervisor/worker** pattern routes each question to the best-qualified worker agent. The supervisor's sole job is routing — it reads the question and decides which worker to invoke.

In [ ]:
class RoutingDecision(BaseModel):
    worker: Literal["filings", "calculations"]
    reasoning: str


class MultiAgentState(TypedDict):
    question: str
    route: str
    filings_answer: str
    calculations_answer: str
    final_answer: str


# Stub filing data
FILING_STORE = {
    "cet1": "CET1 ratio: 14.8% vs 4.5% regulatory minimum.",
    "revenue": "Net revenues: $47.4B, up 8% YoY.",
    "lcr": "LCR: 128% vs 100% regulatory minimum.",
}


def supervisor_node(state: MultiAgentState) -> dict:
    """Classify the question and route to the appropriate worker."""
    messages = [
        {
            "role": "system",
            "content": (
                "Classify which worker should handle this financial question. "
                "'filings' handles SEC filing data, revenues, capital ratios. "
                "'calculations' handles VaR, bond pricing, ratio arithmetic."
            ),
        },
        {"role": "user", "content": state["question"]},
    ]
    decision: RoutingDecision = llm.complete(messages, response_format=RoutingDecision)
    return {"route": decision.worker}


def filings_worker_node(state: MultiAgentState) -> dict:
    """Answer questions from filing data."""
    context = "\n".join(FILING_STORE.values())
    messages = [
        {"role": "system", "content": f"Answer from filings data only:\n{context}"},
        {"role": "user", "content": state["question"]},
    ]
    answer = llm.complete(messages)
    return {"filings_answer": answer, "final_answer": answer}


def calculations_worker_node(state: MultiAgentState) -> dict:
    """Answer questions requiring financial calculations."""
    from scipy.stats import norm
    messages = [
        {
            "role": "system",
            "content": (
                "You are a quantitative analyst. Perform financial calculations precisely. "
                "For VaR: VaR = portfolio_value * volatility * z_alpha where z_0.95=1.645, z_0.99=2.326."
            ),
        },
        {"role": "user", "content": state["question"]},
    ]
    answer = llm.complete(messages)
    return {"calculations_answer": answer, "final_answer": answer}


def route_to_worker(state: MultiAgentState) -> str:
    return "filings_worker" if state["route"] == "filings" else "calculations_worker"


multi_builder = StateGraph(MultiAgentState)
multi_builder.add_node("supervisor", supervisor_node)
multi_builder.add_node("filings_worker", filings_worker_node)
multi_builder.add_node("calculations_worker", calculations_worker_node)
multi_builder.set_entry_point("supervisor")
multi_builder.add_conditional_edges(
    "supervisor",
    route_to_worker,
    {"filings_worker": "filings_worker", "calculations_worker": "calculations_worker"},
)
multi_builder.add_edge("filings_worker", END)
multi_builder.add_edge("calculations_worker", END)
multi_graph = multi_builder.compile()

# Run two questions
for question in [
    "What is our CET1 capital ratio versus the regulatory minimum?",
    "What is the 95% VaR for a $50M portfolio with 12% annual volatility?",
]:
    result = multi_graph.invoke({"question": question, "route": "", "filings_answer": "", "calculations_answer": "", "final_answer": ""})
    print(f"Q: {question}")
    print(f"Route: {result['route']}")
    print(f"A: {result['final_answer']}\n")

## Human-in-the-Loop

LangGraph's `interrupt_before` parameter pauses the graph before the named node and serializes the state to disk. This enables human review: an operator can inspect the classify output, add notes, and resume the graph — or reject the document and stop processing. In financial services compliance, high-risk reviews often require a human signoff before the report is finalized for regulatory purposes.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Recompile with checkpointing and interrupt
checkpointer = MemorySaver()  # <1>
interrupt_graph = compliance_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["review_high_risk"],  # <2>
)

THREAD_ID = "review-001"
config = {"configurable": {"thread_id": THREAD_ID}}

print("--- Phase 1: run to interrupt point ---")
state_at_interrupt = interrupt_graph.invoke(initial_state, config=config)
print(f"Paused before review_high_risk.")
print(f"Doc type: {state_at_interrupt['doc_type']}")
print(f"Risk tier: {state_at_interrupt['risk_tier']}")
print("[HUMAN]: Reviewing classify output... approved to proceed.")

print("\n--- Phase 2: resume from checkpoint ---")
final_state_resumed = interrupt_graph.invoke(None, config=config)  # <3>
print(f"Review complete. Recommendation: {final_state_resumed['report'].recommendation}")

1. `MemorySaver` checkpoints state in memory. In production, use `SqliteSaver` or `PostgresSaver` so state persists across service restarts — the whole point of checkpointing is surviving failures.
2. `interrupt_before=["review_high_risk"]` tells LangGraph to pause execution and return the current state before entering that node. The graph invocation returns immediately with the state as it was when the interrupt triggered.
3. Passing `None` as the input to `invoke` resumes from the checkpoint at `thread_id="review-001"`. The graph picks up exactly where it left off — in this case, entering `review_high_risk` with the full pre-interrupt state.

## Agent Evaluation

We evaluate the compliance pipeline on 5 test documents, using LLM-as-judge to score whether `flagged_items` is complete relative to the expected issues. This mirrors the eval harness from [notebook 04](/courses/llm-eng/04-eval-concepts.html).

In [ ]:
class FlaggedItemsScore(BaseModel):
    score: int  # 1-5
    reasoning: str


TEST_DOCUMENTS = [
    {
        "text": "NDA with mutual confidentiality, standard 2-year term, New York governing law, no unusual provisions.",
        "expected_risk": "low",
        "expected_issues": [],
    },
    {
        "text": "Loan agreement waiving all class action rights, unlimited consequential damages indemnification, no governing law clause.",
        "expected_risk": "high",
        "expected_issues": ["class action waiver", "unlimited indemnification", "missing governing law"],
    },
    {
        "text": "Derivatives master agreement referencing ISDA 2002 schedule, standard credit support annex, New York law.",
        "expected_risk": "medium",
        "expected_issues": ["missing ISDA schedule annex"],
    },
    {
        "text": "Equity compensation plan with standard vesting, no change-of-control provisions, SEC S-8 filed.",
        "expected_risk": "low",
        "expected_issues": [],
    },
    {
        "text": "Margin lending agreement with no margin call notice period, unilateral liquidation rights, no cure period.",
        "expected_risk": "high",
        "expected_issues": ["no margin call notice", "no cure period", "unilateral liquidation"],
    },
]

risk_matches = 0
judge_scores = []

for test in TEST_DOCUMENTS:
    state = {
        "document_text": test["text"],
        "doc_type": "",
        "risk_tier": "",
        "retrieved_policies": [],
        "review_text": "",
        "report": None,
    }
    result = compliance_graph.invoke(state)
    report = result["report"]

    risk_match = result["risk_tier"] == test["expected_risk"]
    risk_matches += int(risk_match)

    # Score flagged items completeness with LLM judge
    judge_messages = [
        {"role": "system", "content": "Score 1-5 whether flagged_items covers all expected compliance issues. 5=complete, 1=major gaps."},
        {"role": "user", "content": f"Expected issues: {test['expected_issues']}\nFlagged items: {report.flagged_items}"},
    ]
    score_result: FlaggedItemsScore = llm.complete(judge_messages, response_format=FlaggedItemsScore)
    judge_scores.append(score_result.score)

    print(f"Doc: {test['text'][:50]}...")
    print(f"  risk: expected={test['expected_risk']} got={result['risk_tier']} match={risk_match}")
    print(f"  flagged: {report.flagged_items}")
    print(f"  judge score: {score_result.score}/5\n")

print(f"Risk tier accuracy: {risk_matches}/{len(TEST_DOCUMENTS)} = {risk_matches/len(TEST_DOCUMENTS):.0%}")
print(f"Avg flagged items score: {sum(judge_scores)/len(judge_scores):.2f}/5")

## Exercises

1. **Add a timeout node.** Create a `check_timeout_node` that inspects a `step_count` field in the state and routes to a `fallback_node` (which returns a default report) if `step_count > 5`. Add it as a conditional edge after `classify`.

2. **Implement parallel retrieval.** Modify `MultiAgentState` to include both `filings_answer` and `calculations_answer` as non-empty fields for questions that need both workers. Add a `merge_node` that combines both answers into `final_answer`. Use LangGraph's `Send` API to invoke both workers concurrently.

3. **Add a `confidence_score` field.** Extend `ComplianceReport` with `confidence_score: float` (0.0–1.0). Modify the review nodes to ask the model to estimate its confidence. Modify the supervisor in the multi-agent graph to route to `interrupt_before=["review_high_risk"]` when `confidence_score < 0.7`, requiring human review.

---

$\blacksquare$